<a href="https://colab.research.google.com/github/Nikhil-N-18/AgenticAI/blob/main/Untitled1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
!pip install -q langchain langchain-groq

In [10]:
import os
from google.colab import userdata

# Get API key securely from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# Import LangChain Groq
from langchain_groq import ChatGroq

# Initialize AI model
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)

print("Groq connected successfully!")

Groq connected successfully!


In [18]:
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=1.0
)

chain = prompt | llm | StrOutputParser()

print("Temperature changed to 1.0")

Temperature changed to 1.0


In [12]:
SYSTEM = """
ROLE:
You are a friendly customer support assistant
for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80 each.
- All cakes have eggless versions.
- Orders require 24 hours notice.

RULES:
- Never invent product prices.
- Never promise delivery times or locations.
- For information not listed in FACTS, reply:
  "I'll check with the owner."
- Calculate totals correctly.
- For multiple cupcakes, multiply Rs 80 by quantity.

FORMAT:
- Keep answers under 50 words.
- Use simple, friendly English.
"""


ROLE:
You are a friendly customer support assistant
for Sweet Crumbs Home Bakery.

FACTS:
- Plum cake costs Rs 500.
- Chocolate cake costs Rs 650.
- Cupcake costs Rs 80 each.
- All cakes have eggless versions.
- Orders require 24 hours notice.

RULES:
- Never invent product prices.
- Never promise delivery times.
- If information is unavailable, say:
  "I'll check with the owner."
- Calculate totals correctly.

FORMAT:
- Keep answers under 50 words.
- Use simple English.
- Be friendly and helpful.



In [13]:
chain = prompt | llm | StrOutputParser()

print("LangChain chain created successfully!")

LangChain chain created successfully!


In [14]:
from langchain_core.prompts import (
    ChatPromptTemplate,
    MessagesPlaceholder
)

from langchain_core.output_parsers import StrOutputParser

# Create prompt template
prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM),
    MessagesPlaceholder("history"),
    ("human", "{question}")
])

print("Prompt template created successfully!")

Prompt template created successfully!


In [15]:
history = []

def ask(question: str) -> str:
    answer = chain.invoke({
        "history": history,
        "question": question
    })

    history.extend([
        ("human", question),
        ("ai", answer)
    ])

    return answer

print(ask("Hi, I am Ravi. Do you make eggless cakes?"))

print(ask("What is my name?"))

Hi Ravi! Yes, all our cakes—including plum, chocolate, and cupcakes—have eggless versions. Let me know what you’d like to order!
Your name is Ravi.


In [16]:
%%writefile tests.csv
question,must_include
What cakes are available?,plum cake
What is the price of chocolate cake?,650
Do you have eggless cakes?,eggless
Can I order a cake for tomorrow?,24 hours
Do you provide home delivery?,check with the owner
What is the price of six cupcakes?,480
Can I customize a birthday cake?,check with the owner
What flavours are available?,check with the owner
How can I place an order?,check with the owner
Can I cancel or modify my order?,check with the owner

Writing tests.csv


In [17]:
import csv

def run_tests(path="tests.csv"):
    with open(path, newline="", encoding="utf-8") as file:
        tests = list(csv.DictReader(file))

    passed = 0

    for test in tests:
        answer = chain.invoke({
            "history": [],
            "question": test["question"]
        })

        expected = test["must_include"]
        success = expected.lower() in answer.lower()

        if success:
            passed += 1
            status = "PASS"
        else:
            status = "FAIL"

        print(f"{status} | {test['question']}")
        print(f"Answer: {answer}\n")

    total = len(tests)
    print(f"Result: {passed}/{total} passed ({passed / total:.0%})")

run_tests()

PASS | What cakes are available?
Answer: We have three cakes:  
- Plum cake – Rs 500  
- Chocolate cake – Rs 650  
- Cupcake – Rs 80 each  

All of them are available in eggless versions. Let me know if you’d like to order!

PASS | What is the price of chocolate cake?
Answer: The chocolate cake costs Rs 650.

PASS | Do you have eggless cakes?
Answer: Yes! All our cakes – plum, chocolate, and cupcakes – come in eggless versions.

FAIL | Can I order a cake for tomorrow?
Answer: I’m sorry, but we need a 24‑hour notice for orders. If you place the order now, we can try to fit it for tomorrow, but I can’t guarantee it. Let me know the cake type and quantity, and I’ll check availability.

PASS | Do you provide home delivery?
Answer: I’ll check with the owner and let you know.

PASS | What is the price of six cupcakes?
Answer: The price for six cupcakes is Rs 480.

FAIL | Can I customize a birthday cake?
Answer: Yes, we can customize a birthday cake. Let us know the size, flavor, and design d

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=SYSTEM
)

print("Agent created successfully!")

In [ ]:
result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Do you make eggless cakes?"
        }
    ]
})

print(result["messages"][-1].content)

In [ ]:
MENU = {
    "plum cake": 500,
    "chocolate cake": 650,
    "cupcake": 80
}

def get_price(item: str, qty: int = 1) -> int:
    """Return the total price in rupees for an item and quantity."""
    item = item.lower().strip()

    if item not in MENU:
        raise ValueError("Item not found in menu")

    if qty < 1:
        raise ValueError("Quantity must be at least 1")

    return MENU[item] * qty

print(get_price("cupcake", 6))
print(get_price("plum cake"))

In [ ]:
def is_available(item: str, hours_notice: int) -> bool:
    """Check whether a menu item can be ordered with the given notice."""
    item = item.lower().strip()

    if item not in MENU:
        return False

    return hours_notice >= 24

print(is_available("plum cake", 48))
print(is_available("chocolate cake", 5))